In [9]:
%pip install -q rdflib

Note: you may need to restart the kernel to use updated packages.


In [11]:
import sys; sys.path.append("/home/jovyan/work")
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, OWL, XSD

PARIS =  Namespace("http://example.org/comptage-multimodal#") # noms du notre projet
g = Graph()
g.bind("", PARIS); g.bind("owl", OWL); g.bind("rdfs", RDFS)


g.parse(
    "/home/jovyan/work/semantique/ontologie_comptagesparis.ttl",
    format="turtle"
)

print("Triples de l'ontologie :", len(g))

Nombre de triples : 46


## Charger un extrait réel de Bronze dans le graphe RDF

RDFLib conserve le graphe en mémoire. Pour éviter d'y charger les millions de lignes, le notebook lit le journal Delta de RustFS et charge dans PyArrow au plus 200 observations de janvier 2025. Les requêtes métier ci-dessous portent donc sur cet extrait réel, et non sur l'ensemble du dataset. Bronze est lu sans réécriture ni modification.

In [ ]:
import io
import json
import os
from datetime import datetime
from urllib.parse import quote
from zoneinfo import ZoneInfo

import boto3
import pyarrow.parquet as pq

BUCKET = "datalake"
BRONZE_PREFIX = "bronze/comptage_multimodal_paris"
MAX_OBSERVATIONS = 200
s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("S3_ENDPOINT", "http://rustfs:9000"),
    aws_access_key_id=os.getenv("S3_ACCESS_KEY", "rustfsadmin"),
    aws_secret_access_key=os.getenv("S3_SECRET_KEY", "rustfsadmin"),
    region_name="us-east-1",
)

journal_delta = []
for page in s3.get_paginator("list_objects_v2").paginate(
    Bucket=BUCKET, Prefix=f"{BRONZE_PREFIX}/_delta_log/"
):
    journal_delta.extend(
        item["Key"] for item in page.get("Contents", [])
        if item["Key"].rsplit("/", 1)[-1].endswith(".json")
    )
journal_delta.sort()
if not journal_delta:
    raise FileNotFoundError(f"Aucun journal Delta trouvé pour s3://{BUCKET}/{BRONZE_PREFIX}")

fichiers_actifs = {}
for cle_journal in journal_delta:
    contenu = s3.get_object(Bucket=BUCKET, Key=cle_journal)["Body"].read().decode("utf-8")
    for evenement in contenu.splitlines():
        action = json.loads(evenement)
        if "add" in action:
            fichier = action["add"]["path"]
            fichiers_actifs[fichier] = None
        elif "remove" in action:
            fichiers_actifs.pop(action["remove"]["path"], None)
if not fichiers_actifs:
    raise RuntimeError("Le journal Delta Bronze ne référence aucun fichier actif.")

colonnes_bronze = [
    "identifiant_site", "nom_du_site", "date_heure_comptage",
    "mode_deplacement", "nombre_de_vehicules", "type_voie",
    "sens_circulation", "coordonnees_geo",
]
debut_janvier = datetime(2025, 1, 1, tzinfo=ZoneInfo("Europe/Paris"))
debut_fevrier = datetime(2025, 2, 1, tzinfo=ZoneInfo("Europe/Paris"))
observations_janvier = []
for fichier in sorted(fichiers_actifs):
    cle_objet = f"{BRONZE_PREFIX}/{fichier}"
    contenu = s3.get_object(Bucket=BUCKET, Key=cle_objet)["Body"].read()
    table = pq.read_table(io.BytesIO(contenu), columns=colonnes_bronze, use_threads=False)
    for ligne in table.to_pylist():
        if not ligne["identifiant_site"] or not ligne["mode_deplacement"]:
            continue
        try:
            date_heure = datetime.fromisoformat(str(ligne["date_heure_comptage"]).replace("Z", "+00:00"))
            if date_heure.tzinfo is None:
                date_heure = date_heure.replace(tzinfo=ZoneInfo("Europe/Paris"))
            nombre = int(str(ligne["nombre_de_vehicules"]).strip())
        except (TypeError, ValueError):
            continue
        if debut_janvier <= date_heure < debut_fevrier:
            ligne["date_heure"] = date_heure
            ligne["nombre"] = nombre
            observations_janvier.append(ligne)
            if len(observations_janvier) == MAX_OBSERVATIONS:
                break
    if len(observations_janvier) == MAX_OBSERVATIONS:
        break

if not observations_janvier:
    raise RuntimeError("Aucune observation exploitable trouvée dans Bronze pour janvier 2025.")
observations_janvier.sort(
    key=lambda ligne: (ligne["date_heure"], ligne["identifiant_site"], ligne["mode_deplacement"]),
)

def uri_ressource(type_ressource, identifiant):
    return URIRef(f"{PARIS}{type_ressource}/{quote(str(identifiant), safe='')}")

for index, ligne in enumerate(observations_janvier, start=1):
    observation = uri_ressource("observation", f"janvier-2025-{index:04d}")
    site = uri_ressource("site", ligne["identifiant_site"])
    mode = uri_ressource("mode", ligne["mode_deplacement"])

    g.add((observation, RDF.type, PARIS.ObservationComptage))
    g.add((observation, PARIS.concerneSite, site))
    g.add((observation, PARIS.concerneMode, mode))
    g.add((observation, PARIS.dateHeureComptage, Literal(ligne["date_heure"].isoformat(), datatype=XSD.dateTime)))
    g.add((observation, PARIS.nombreVehicules, Literal(int(ligne["nombre"]), datatype=XSD.integer)))
    g.add((site, RDF.type, PARIS.SiteComptage))
    g.add((site, PARIS.identifiantSite, Literal(str(ligne["identifiant_site"]), datatype=XSD.string)))
    if ligne["nom_du_site"]:
        g.add((site, PARIS.nomSite, Literal(str(ligne["nom_du_site"]), datatype=XSD.string)))
    if ligne["coordonnees_geo"]:
        g.add((site, PARIS.coordonneesGeo, Literal(str(ligne["coordonnees_geo"]), datatype=XSD.string)))

    g.add((mode, RDF.type, PARIS.ModeDeplacement))
    g.add((mode, PARIS.nomMode, Literal(str(ligne["mode_deplacement"]), datatype=XSD.string)))
    if ligne["type_voie"]:
        type_voie = uri_ressource("type-voie", ligne["type_voie"])
        g.add((observation, PARIS.concerneTypeVoie, type_voie))
        g.add((type_voie, RDF.type, PARIS.TypeVoie))
        g.add((type_voie, RDFS.label, Literal(str(ligne["type_voie"]), lang="fr")))
    if ligne["sens_circulation"]:
        g.add((observation, PARIS.sensCirculation, Literal(str(ligne["sens_circulation"]), datatype=XSD.string)))

print(f"Observations réelles chargées depuis Bronze : {len(observations_janvier)} (maximum {MAX_OBSERVATIONS})")
print(f"Triples dans le graphe RDF, ontologie comprise : {len(g)}")

In [12]:

requete = """
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT ?classe
WHERE {
    ?classe a owl:Class .
}
"""

resultats = g.query(requete)

for resultat in resultats:
    print(resultat.classe)


http://example.org/comptage-multimodal#ObservationComptage
http://example.org/comptage-multimodal#SiteComptage
http://example.org/comptage-multimodal#ModeDeplacement
http://example.org/comptage-multimodal#TypeVoie


In [13]:

requete = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT ?propriete
WHERE {
    ?propriete a owl:ObjectProperty .
}
"""

resultats = g.query(requete)

for resultat in resultats:
    print(resultat.propriete)


http://example.org/comptage-multimodal#concerneSite
http://example.org/comptage-multimodal#concerneMode
http://example.org/comptage-multimodal#concerneTypeVoie


In [ ]:
# Requête 1 — modes présents dans l'extrait Bronze chargé
requete_modes = """
PREFIX : <http://example.org/comptage-multimodal#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?mode ?nom
WHERE {
    ?observation rdf:type :ObservationComptage .
    ?observation :concerneMode ?mode .
    ?mode :nomMode ?nom .
}
ORDER BY ?nom
"""
resultats_modes = list(g.query(requete_modes))
assert resultats_modes, "La requête SPARQL des modes n'a retourné aucun résultat."
print(f"Modes trouvés dans l'extrait : {len(resultats_modes)}")

for resultat in resultats_modes:
    print(resultat.nom)


NameError: name 'g' is not defined

In [42]:
# Requête 2 — sites présents dans l'extrait Bronze chargé
requete_sites = """
PREFIX : <http://example.org/comptage-multimodal#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT DISTINCT ?identifiant ?nom
WHERE {
    ?site rdf:type :SiteComptage .
    ?site :identifiantSite ?identifiant .
    ?site :nomSite ?nom .
}
ORDER BY ?nom
"""
resultats_sites = list(g.query(requete_sites))
assert resultats_sites, "La requête SPARQL des sites n'a retourné aucun résultat."
print(f"Sites trouvés dans l'extrait : {len(resultats_sites)} ; affichage des 10 premiers")

for resultat in resultats_sites[:10]:
    print(resultat.identifiant, resultat.nom)

In [43]:
# Requête 3 — comptages de janvier 2025, triés par date
requete_observations_janvier = """
PREFIX : <http://example.org/comptage-multimodal#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?observation ?date ?nombre ?modeNom ?identifiantSite ?voie
WHERE {
    ?observation rdf:type :ObservationComptage .
    ?observation :dateHeureComptage ?date .
    ?observation :nombreVehicules ?nombre .
    ?observation :concerneMode ?mode .
    ?mode :nomMode ?modeNom .
    ?observation :concerneSite ?site .
    ?site :identifiantSite ?identifiantSite .
    OPTIONAL {
        ?observation :concerneTypeVoie ?typeVoie .
        ?typeVoie rdfs:label ?voie .
    }

    FILTER (
        ?date >= "2025-01-01T00:00:00"^^xsd:dateTime &&
        ?date < "2025-02-01T00:00:00"^^xsd:dateTime
    )
}
ORDER BY ?date
LIMIT 20
"""
resultats_janvier = list(g.query(requete_observations_janvier))
assert resultats_janvier, "La requête SPARQL de janvier 2025 n'a retourné aucun résultat."
print(f"Observations de janvier retournées : {len(resultats_janvier)} (maximum 20 affichées)")

for resultat in resultats_janvier:
    print(resultat.date, resultat.modeNom, resultat.nombre, resultat.identifiantSite, resultat.voie)

NameError: name 'g' is not defined

In [ ]:
# Créer un graphe de test et y charger l'ontologie -- testing
g_test = Graph()
g_test.parse("/home/jovyan/work/semantique/ontologie_comptagesparis.ttl",
             format="turtle")

PARIS = Namespace("http://example.org/comptage-multimodal#")

# Ajouter deux modes de déplacement
g_test.add((
    PARIS.observation_test_1,
    RDF.type,
    PARIS.ObservationComptage
))
g_test.add((
    PARIS.observation_test_1,
    PARIS.concerneMode,
    PARIS.mode_trottinette
))
g_test.add((
    PARIS.observation_test_2,
    RDF.type,
    PARIS.ObservationComptage
))
g_test.add((
    PARIS.observation_test_2,
    PARIS.concerneMode,
    PARIS.mode_velo
))

# Ajouter les noms des modes pour les afficher
g_test.add((
    PARIS.mode_trottinette,
    RDFS.label,
    Literal("Trottinettes", lang="fr")
))
g_test.add((
    PARIS.mode_velo,
    RDFS.label,
    Literal("Vélos", lang="fr")
))

print("Nombre de triples dans le graphe de test :", len(g_test))


NameError: name 'Graph' is not defined

In [47]:

requete_test = """
PREFIX : <http://example.org/comptage-multimodal#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?mode ?nom
WHERE {
    ?observation rdf:type :ObservationComptage .
    ?observation :concerneMode ?mode .
    ?mode rdfs:label ?nom .
}
ORDER BY ?nom
"""

for resultat in g_test.query(requete_test):
    print(resultat)


(rdflib.term.URIRef('http://example.org/comptage-multimodal#mode_trottinette'), rdflib.term.Literal('Trottinettes', lang='fr'))
(rdflib.term.URIRef('http://example.org/comptage-multimodal#mode_velo'), rdflib.term.Literal('Vélos', lang='fr'))
